# E7 - Inventario SSMSpine / SymTC

Este notebook inventaria SSMSpine/SymTC como posible dataset adicional para segmentacion lumbar. La hipotesis inicial es que SSMSpine parece sintetico y mid-sagittal, no axial. El objetivo es verificarlo tecnicamente con inventario, lectura `.pt`, visualizaciones, analisis de labels y una decision conservadora para el proyecto.

No se entrenan modelos en este notebook.

In [ ]:
try:
    import torch  # noqa: F401
except Exception:
    %pip install -q torch

In [ ]:
import json
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 160)

## Montaje de Drive y rutas

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

In [ ]:
SSMSPINE_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SSMSPINE")
RAW_ZIPS = SSMSPINE_ROOT / "raw_zips"
EXTRACTED_ROOT = SSMSPINE_ROOT / "extracted"
PROCESSED_ROOT = SSMSPINE_ROOT / "processed"
RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E7_ssmspine_inventory")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [RAW_ZIPS, EXTRACTED_ROOT, PROCESSED_ROOT, RESULTS_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_SAMPLE_ZIPS = ["SSMSpine_Train_Sample.zip", "SSMSpine_Test_Sample.zip"]
EXPECTED_FULL_ZIPS = ["SSMSpine_Train.zip", "SSMSpine_Test.zip"]

EXTRACT_SAMPLE_ZIPS = True
EXTRACT_FULL_DATASET = False
FORCE_EXTRACT = False
PT_SAMPLE_N = 20
VIS_SAMPLE_N = 10
LABEL_SAMPLE_N = 100
RANDOM_SEED = 42

print("SSMSPINE_ROOT:", SSMSPINE_ROOT)
print("RAW_ZIPS:", RAW_ZIPS)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)

## Verificacion de ZIPs

In [ ]:
def human_size(num_bytes):
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:.2f} {unit}"
        value /= 1024


zip_names_to_check = EXPECTED_SAMPLE_ZIPS + EXPECTED_FULL_ZIPS
zip_rows = []
for name in zip_names_to_check:
    path = RAW_ZIPS / name
    detected = path.exists()
    zip_rows.append({
        "zip_name": name,
        "zip_path": str(path),
        "detected": bool(detected),
        "size_bytes": int(path.stat().st_size) if detected else 0,
        "size_human": human_size(path.stat().st_size) if detected else "0 B",
        "expected_sample": name in EXPECTED_SAMPLE_ZIPS,
        "expected_full": name in EXPECTED_FULL_ZIPS,
    })

for path in sorted(RAW_ZIPS.glob("*.zip")):
    if path.name not in zip_names_to_check:
        zip_rows.append({
            "zip_name": path.name,
            "zip_path": str(path),
            "detected": True,
            "size_bytes": int(path.stat().st_size),
            "size_human": human_size(path.stat().st_size),
            "expected_sample": False,
            "expected_full": False,
        })

zip_files_df = pd.DataFrame(zip_rows)
zip_files_csv_path = RESULTS_ROOT / "E7_ssmspine_zip_files.csv"
zip_files_df.to_csv(zip_files_csv_path, index=False)
display(zip_files_df)

missing_samples = zip_files_df[zip_files_df["expected_sample"] & ~zip_files_df["detected"]]["zip_name"].tolist()
if missing_samples:
    print("ADVERTENCIA: faltan ZIPs sample esperados:", missing_samples)

## Inventario interno de ZIPs

In [ ]:
def count_zip_extensions(zip_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        infos = [info for info in zf.infolist() if not info.is_dir()]
        names = [info.filename for info in infos]
    counts = {".pt": 0, ".png": 0, ".json": 0, ".csv": 0, "other": 0}
    for name in names:
        ext = Path(name).suffix.lower()
        if ext in counts:
            counts[ext] += 1
        else:
            counts["other"] += 1
    all_text = " ".join(names).lower()
    return {
        "total_files": len(names),
        "count_pt": counts[".pt"],
        "count_png": counts[".png"],
        "count_json": counts[".json"],
        "count_csv": counts[".csv"],
        "count_other": counts["other"],
        "example_internal_paths": json.dumps(names[:25], ensure_ascii=False),
        "has_train": "train" in all_text,
        "has_test": "test" in all_text,
        "has_sample": "sample" in all_text,
    }


zip_inventory_rows = []
for _, row in zip_files_df[zip_files_df["detected"]].iterrows():
    stats = count_zip_extensions(Path(row["zip_path"]))
    zip_inventory_rows.append({"zip_name": row["zip_name"], **stats})

zip_inventory_df = pd.DataFrame(zip_inventory_rows)
zip_inventory_csv_path = RESULTS_ROOT / "E7_ssmspine_zip_inventory.csv"
zip_inventory_df.to_csv(zip_inventory_csv_path, index=False)
display(zip_inventory_df)

## Extraccion controlada

In [ ]:
def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if not str(target).startswith(str(destination)):
                raise RuntimeError(f"Ruta insegura en ZIP: {member.filename}")
        zf.extractall(destination)


extract_rows = []
for _, row in zip_files_df[zip_files_df["detected"]].iterrows():
    name = row["zip_name"]
    should_extract = (name in EXPECTED_SAMPLE_ZIPS and EXTRACT_SAMPLE_ZIPS) or (name in EXPECTED_FULL_ZIPS and EXTRACT_FULL_DATASET)
    destination = EXTRACTED_ROOT / Path(name).stem
    destination.mkdir(parents=True, exist_ok=True)
    existing_files = [p for p in destination.rglob("*") if p.is_file()]
    skipped_existing = bool(existing_files and not FORCE_EXTRACT)
    error = None
    extracted_count = len(existing_files)
    if should_extract:
        try:
            if FORCE_EXTRACT and destination.exists():
                import shutil
                shutil.rmtree(destination)
                destination.mkdir(parents=True, exist_ok=True)
            if not skipped_existing:
                safe_extract_zip(Path(row["zip_path"]), destination)
                extracted_count = sum(1 for p in destination.rglob("*") if p.is_file())
        except Exception as exc:
            error = repr(exc)
    extract_rows.append({
        "zip_name": name,
        "destination": str(destination),
        "should_extract": bool(should_extract),
        "skipped_existing": bool(skipped_existing),
        "force_extract": FORCE_EXTRACT,
        "extracted_file_count": int(extracted_count),
        "error": error,
    })

extraction_report_df = pd.DataFrame(extract_rows)
extraction_report_csv_path = RESULTS_ROOT / "E7_ssmspine_extraction_report.csv"
extraction_report_df.to_csv(extraction_report_csv_path, index=False)
display(extraction_report_df)

## Inventario extraido

In [ ]:
def split_candidate_from_path(path):
    text = str(path).lower()
    if "train" in text:
        return "train"
    if "test" in text:
        return "test"
    return "unknown"


def sample_or_full_from_path(path):
    return "sample" if "sample" in str(path).lower() else "full"


file_rows = []
for path in tqdm([p for p in EXTRACTED_ROOT.rglob("*") if p.is_file()], desc="Inventario extraido"):
    rel = path.relative_to(EXTRACTED_ROOT)
    parts = rel.parts
    patient_folder = next((part for part in parts if any(ch.isdigit() for ch in part)), None)
    aug = next((part for part in parts if "aug" in part.lower() or "crop" in part.lower()), None)
    file_rows.append({
        "file_path": str(path),
        "relative_path": str(rel),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "parent_folder": path.parent.name,
        "size_bytes": int(path.stat().st_size),
        "split_candidate": split_candidate_from_path(rel),
        "sample_or_full": sample_or_full_from_path(rel),
        "patient_folder_candidate": patient_folder,
        "augmentation_candidate": aug,
    })

extracted_inventory_df = pd.DataFrame(file_rows)
extracted_inventory_csv_path = RESULTS_ROOT / "E7_ssmspine_extracted_file_inventory.csv"
extracted_inventory_df.to_csv(extracted_inventory_csv_path, index=False)
print("Archivos extraidos:", len(extracted_inventory_df))
display(extracted_inventory_df.head())

## Lectura de archivos PT

In [ ]:
def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    if isinstance(value, np.ndarray):
        return value
    return None


def summarize_pt_value(value):
    arr = to_numpy(value)
    if arr is None:
        return {"shape": None, "dtype": str(type(value)), "min": None, "max": None}
    numeric = np.issubdtype(arr.dtype, np.number)
    return {
        "shape": str(tuple(arr.shape)),
        "dtype": str(arr.dtype),
        "min": float(np.nanmin(arr)) if numeric and arr.size else None,
        "max": float(np.nanmax(arr)) if numeric and arr.size else None,
    }


pt_files_df = extracted_inventory_df[extracted_inventory_df["extension"].eq(".pt")].copy()
pt_sample_df = pt_files_df.sample(n=min(PT_SAMPLE_N, len(pt_files_df)), random_state=RANDOM_SEED) if len(pt_files_df) else pd.DataFrame()

pt_summary_rows = []
loaded_samples = []
for _, row in tqdm(pt_sample_df.iterrows(), total=len(pt_sample_df), desc="Leyendo PT sample"):
    item = {
        "file_path": row["file_path"],
        "relative_path": row["relative_path"],
        "read_ok": False,
        "error": None,
    }
    try:
        data = torch.load(row["file_path"], map_location="cpu")
        keys = list(data.keys()) if isinstance(data, dict) else []
        item["read_ok"] = True
        item["keys"] = json.dumps(keys)
        for key in keys:
            s = summarize_pt_value(data[key])
            item[f"{key}_shape"] = s["shape"]
            item[f"{key}_dtype"] = s["dtype"]
            item[f"{key}_min"] = s["min"]
            item[f"{key}_max"] = s["max"]
        for k in ["img", "label", "mask", "bone_mask", "disk_mask"]:
            item[f"has_{k}"] = k in keys
        item["has_landmarks"] = any("landmark" in str(k).lower() for k in keys)
        loaded_samples.append((row.to_dict(), data))
    except Exception as exc:
        item["error"] = repr(exc)
    pt_summary_rows.append(item)

pt_sample_summary_df = pd.DataFrame(pt_summary_rows)
pt_sample_summary_csv_path = RESULTS_ROOT / "E7_ssmspine_pt_sample_summary.csv"
pt_sample_summary_df.to_csv(pt_sample_summary_csv_path, index=False)
display(pt_sample_summary_df.head())

## Validacion de estructura esperada

In [ ]:
expected_shapes = {
    "img": (512, 512),
    "img_corrected": (512, 512),
    "img_equalized": (512, 512),
    "label": (1, 512, 512),
    "mask": (11, 512, 512),
    "bone_mask": (6, 512, 512),
    "disk_mask": (5, 512, 512),
}

schema_rows = []
for meta, data in loaded_samples:
    row = {"file_path": meta["file_path"], "relative_path": meta["relative_path"]}
    for key, expected in expected_shapes.items():
        present = isinstance(data, dict) and key in data
        arr = to_numpy(data[key]) if present else None
        row[f"{key}_present"] = present
        row[f"{key}_shape"] = str(tuple(arr.shape)) if arr is not None else None
        row[f"{key}_matches_expected"] = bool(arr is not None and tuple(arr.shape) == expected)
    schema_rows.append(row)

schema_validation_df = pd.DataFrame(schema_rows)
schema_validation_csv_path = RESULTS_ROOT / "E7_ssmspine_schema_validation.csv"
schema_validation_df.to_csv(schema_validation_csv_path, index=False)
display(schema_validation_df.head())

## Visualizacion de muestras

In [ ]:
def squeeze_first(arr):
    arr = np.asarray(arr)
    while arr.ndim > 2:
        arr = arr[0]
    return arr


def combine_channels(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        return np.max(arr, axis=0)
    return squeeze_first(arr)


def norm_img(arr):
    arr = np.asarray(arr, dtype=np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    if np.isclose(lo, hi):
        return np.zeros_like(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)


figure_paths = []
for idx, (meta, data) in enumerate(loaded_samples[:VIS_SAMPLE_N], start=1):
    img = to_numpy(data.get("img")) if isinstance(data, dict) and "img" in data else None
    label_arr = to_numpy(data.get("label")) if isinstance(data, dict) and "label" in data else None
    disk = to_numpy(data.get("disk_mask")) if isinstance(data, dict) and "disk_mask" in data else None
    bone = to_numpy(data.get("bone_mask")) if isinstance(data, dict) and "bone_mask" in data else None
    if img is None:
        continue
    img2d = squeeze_first(img)
    label2d = squeeze_first(label_arr) if label_arr is not None else np.zeros_like(img2d)
    disk2d = combine_channels(disk) if disk is not None else np.zeros_like(img2d)
    bone2d = combine_channels(bone) if bone is not None else np.zeros_like(img2d)

    fig, axes = plt.subplots(1, 5, figsize=(17, 4))
    axes[0].imshow(norm_img(img2d), cmap="gray")
    axes[0].set_title("img")
    axes[1].imshow(label2d, cmap="tab20")
    axes[1].set_title("label")
    axes[2].imshow(norm_img(img2d), cmap="gray")
    axes[2].imshow(np.ma.masked_where(label2d == 0, label2d), cmap="tab20", alpha=0.45)
    axes[2].set_title("overlay label")
    axes[3].imshow(disk2d, cmap="magma")
    axes[3].set_title("disk mask")
    axes[4].imshow(bone2d, cmap="viridis")
    axes[4].set_title("bone mask")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(Path(meta["relative_path"]).name)
    fig.tight_layout()
    out = FIGURES_ROOT / f"E7_ssmspine_sample_{idx:02d}.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.close(fig)
    figure_paths.append(str(out))

print("Figuras:", figure_paths)

## Analisis de labels

In [ ]:
LABEL_MAPPING = {
    0: "background", 1: "L1", 2: "L2", 3: "L3", 4: "L4", 5: "L5", 6: "S1",
    7: "D1_L1_L2", 8: "D2_L2_L3", 9: "D3_L3_L4", 10: "D4_L4_L5", 11: "D5_L5_S1",
}

label_rows = []
for meta, data in loaded_samples[:LABEL_SAMPLE_N]:
    if not isinstance(data, dict) or "label" not in data:
        continue
    label_np = squeeze_first(to_numpy(data["label"]))
    total = max(label_np.size, 1)
    labels, counts = np.unique(label_np, return_counts=True)
    for label_value, count in zip(labels, counts):
        lv = int(label_value)
        label_rows.append({
            "file_path": meta["file_path"],
            "relative_path": meta["relative_path"],
            "label": lv,
            "label_name": LABEL_MAPPING.get(lv, "unknown"),
            "pixel_count": int(count),
            "pixel_ratio": float(count / total),
            "foreground_ratio": float(np.mean(label_np > 0)),
            "present_expected_0_11": bool(0 <= lv <= 11),
        })

label_distribution_df = pd.DataFrame(label_rows)
label_distribution_csv_path = RESULTS_ROOT / "E7_ssmspine_label_distribution.csv"
label_distribution_df.to_csv(label_distribution_csv_path, index=False)
display(label_distribution_df.head())

## Evaluacion de orientacion y utilidad axial

In [ ]:
keys_found = sorted(set(k for _, data in loaded_samples if isinstance(data, dict) for k in data.keys()))
has_masks = any(k in keys_found for k in ["label", "mask", "bone_mask", "disk_mask"])
all_shapes_2d = True
for _, data in loaded_samples:
    if not isinstance(data, dict) or "img" not in data:
        continue
    arr = to_numpy(data["img"])
    if arr is not None and arr.ndim not in [2, 3]:
        all_shapes_2d = False

appears_2d_synthetic = bool(len(pt_files_df) > 0 and all_shapes_2d and not any(extracted_inventory_df["extension"].isin([".dcm", ".ima"])))
appears_mid_sagittal = bool(has_masks and any(label_distribution_df.get("label", pd.Series(dtype=int)).isin(range(1, 12)))) if len(label_distribution_df) else has_masks
appears_axial = False
has_dicom_metadata = bool(any(extracted_inventory_df["extension"].isin([".dcm", ".ima"])))

if appears_axial:
    recommended_use = "usable_for_axial_segmentation"
elif appears_mid_sagittal and has_masks:
    recommended_use = "usable_for_sagittal_segmentation"
elif has_masks:
    recommended_use = "usable_for_pretraining_or_benchmark"
else:
    recommended_use = "not_useful_for_current_scope"

orientation_assessment = {
    "appears_2d_synthetic": appears_2d_synthetic,
    "appears_mid_sagittal": appears_mid_sagittal,
    "appears_axial": appears_axial,
    "has_dicom_metadata": has_dicom_metadata,
    "has_segmentation_masks": bool(has_masks),
    "recommended_use": recommended_use,
}
orientation_assessment_json_path = RESULTS_ROOT / "E7_ssmspine_orientation_assessment.json"
with open(orientation_assessment_json_path, "w", encoding="utf-8") as f:
    json.dump(orientation_assessment, f, indent=2, ensure_ascii=False)
print(json.dumps(orientation_assessment, indent=2, ensure_ascii=False))

## Reporte JSON y conclusion

In [ ]:
if recommended_use == "usable_for_axial_segmentation":
    technical_decision = "usable_for_axial_segmentation"
elif recommended_use == "usable_for_sagittal_segmentation":
    technical_decision = "usable_for_sagittal_segmentation"
elif recommended_use == "usable_for_pretraining_or_benchmark":
    technical_decision = "usable_for_pretraining_or_benchmark"
else:
    technical_decision = "not_useful_for_current_scope"

report = {
    "zips_detected": zip_files_df.to_dict(orient="records"),
    "extracted_files": int(len(extracted_inventory_df)),
    "count_pt": int(len(pt_files_df)),
    "keys_found": keys_found,
    "n_visualized": int(len(figure_paths)),
    "labels_present": sorted(label_distribution_df["label"].dropna().astype(int).unique().tolist()) if len(label_distribution_df) else [],
    "orientation_inferred": orientation_assessment,
    "technical_decision": technical_decision,
    "recommendation_next_notebooks": (
        "No usar para axial; considerar como dataset sintetico sagital para comparacion/pretraining."
        if not appears_axial else "Evaluar baseline axial solo si se confirma metadata axial adicional."
    ),
    "exports": {
        "zip_files": str(zip_files_csv_path),
        "zip_inventory": str(zip_inventory_csv_path),
        "extraction_report": str(extraction_report_csv_path),
        "extracted_inventory": str(extracted_inventory_csv_path),
        "pt_sample_summary": str(pt_sample_summary_csv_path),
        "schema_validation": str(schema_validation_csv_path),
        "label_distribution": str(label_distribution_csv_path),
        "orientation_assessment": str(orientation_assessment_json_path),
        "figures": figure_paths,
    },
}
report_json_path = RESULTS_ROOT / "E7_ssmspine_inventory_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Reporte:", report_json_path)

In [ ]:
zip_table_md = zip_files_df.to_markdown(index=False)
schema_md = schema_validation_df.head(20).to_markdown(index=False) if len(schema_validation_df) else "No se leyeron muestras PT."
label_md = label_distribution_df.groupby(["label", "label_name"], dropna=False)["pixel_ratio"].mean().reset_index().to_markdown(index=False) if len(label_distribution_df) else "Sin labels leidos."

conclusion_text = f'''# Conclusion tecnica - E7 Inventario SSMSpine/SymTC

## Objetivo

Inventariar SSMSpine/SymTC, leer muestras `.pt`, visualizar imagenes y mascaras, analizar labels y decidir si el dataset sirve para el proyecto, especialmente para el plano axial.

## Dataset evaluado

Repositorio fuente: `https://github.com/jiasongchen/SSMSpine`
Repositorio SymTC: `https://github.com/jiasongchen/SymTC`

## ZIPs evaluados

{zip_table_md}

## Estructura encontrada

Archivos `.pt` inventariados: {len(pt_files_df)}.

{schema_md}

## Labels encontrados

{label_md}

## Ejemplos visuales

{figure_paths}

## Utilidad axial

- Parece 2D sintetico: {orientation_assessment["appears_2d_synthetic"]}.
- Parece mid-sagittal: {orientation_assessment["appears_mid_sagittal"]}.
- Parece axial: {orientation_assessment["appears_axial"]}.
- Tiene metadata DICOM/orientacion: {orientation_assessment["has_dicom_metadata"]}.

Decision tecnica: `{technical_decision}`.

## Recomendacion

Si no se confirma informacion axial, no usar SSMSpine como solucion axial. Puede servir como dataset sintetico sagital, benchmark metodologico o preentrenamiento, separado del modulo axial clinico.
'''

conclusion_path = DOCS_ROOT / "E7_ssmspine_inventory_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)
print("Conclusion:", conclusion_path)

## Criterio de aceptacion

Este notebook detecta ZIPs sample, extrae samples, inventaria `.pt`, intenta cargar muestras con `torch.load`, identifica keys/shapes, visualiza imagen + mascara, analiza labels, decide explicitamente si sirve para axial, exporta CSV/JSON/Markdown y no entrena modelos.